# NearMe Medallion Data Cleaning Pipeline

Reads raw environmental data from the `datastream_ingestion` Delta tables (bronze), cleans and standardizes it (silver), and produces analytics-ready aggregated tables (gold). Sources: EPA Air Quality, NOAA Weather, USGS Water, NCDOT Crash Data.

In [0]:
# Bronze Layer — Read raw Delta tables from datastream_ingestion
# These tables are appended every 20 minutes by the scheduled ingestion job.
# Bronze = raw, unmodified data exactly as ingested.

from pyspark.sql.functions import col, current_timestamp

bronze_tables = {
    "bronze_epa_air_quality": spark.table("workspace.default.epa_air_quality"),
    "bronze_noaa_weather_observations": spark.table("workspace.default.noaa_weather_observations"),
    "bronze_noaa_weather_alerts": spark.table("workspace.default.noaa_weather_alerts"),
    "bronze_usgs_water_data": spark.table("workspace.default.usgs_water_data"),
    "bronze_ncdot_crash_data": spark.table("workspace.default.ncdot_crash_data"),
    "bronze_ncgs_landslide_data": spark.table("workspace.default.ncgs_landslide_data"),
    "bronze_ncem_flood_zones": spark.table("workspace.default.ncem_flood_zones"),
}

print("=" * 60)
print("Bronze Layer — Raw data from ingestion pipeline")
print("=" * 60)
for name, df in bronze_tables.items():
    cnt = df.count()
    print(f"  {name}: {cnt} rows")
    df.createOrReplaceTempView(name)

print("\nBronze layer loaded successfully.")

In [0]:
# Silver Layer — EPA Air Quality Cleaning
# Issues found in bronze:
#   - Duplicate rows: same measurement re-ingested every 20 min
#   - Timestamps stored as strings (measurement_time, ingested_at)
#   - Placeholder rows with all-null pollutants when API fails
#   - No AQI category label (only numeric us_aqi)

from pyspark.sql.functions import col, to_timestamp, trim, when, row_number
from pyspark.sql.window import Window

bronze_epa = spark.table("bronze_epa_air_quality")

silver_epa = (bronze_epa
    # Drop rows where ALL pollutant measurements AND aqi are null (placeholder rows)
    .filter(
        col("us_aqi").isNotNull() |
        col("pm10").isNotNull() |
        col("pm2_5").isNotNull() |
        col("o3").isNotNull() |
        col("no2").isNotNull() |
        col("so2").isNotNull() |
        col("co").isNotNull()
    )
    # Cast timestamps from strings to proper timestamp type
    .withColumn("measurement_time", to_timestamp(col("measurement_time")))
    .withColumn("ingested_at", to_timestamp(col("ingested_at"), "yyyyMMdd_HHmmss"))
    # Deduplicate: keep latest ingestion per measurement_time + lat + lon
    .withColumn("rn", row_number().over(
        Window.partitionBy("measurement_time", "latitude", "longitude")
        .orderBy(col("ingested_at").desc())
    ))
    .filter(col("rn") == 1)
    .drop("rn")
    # Add AQI category derived from numeric us_aqi (EPA breakpoints)
    .withColumn("aqi_category",
        when(col("us_aqi") <= 50, "Good")
        .when(col("us_aqi") <= 100, "Moderate")
        .when(col("us_aqi") <= 150, "Unhealthy for Sensitive Groups")
        .when(col("us_aqi") <= 200, "Unhealthy")
        .when(col("us_aqi") <= 300, "Very Unhealthy")
        .otherwise("Hazardous"))
    # Trim text fields
    .withColumn("source", trim(col("source")))
    .withColumn("category", trim(col("category")))
    .withColumn("parameter", trim(col("parameter")))
)

silver_epa.write.mode("overwrite").saveAsTable("workspace.default.silver_epa_air_quality")
cnt = spark.sql("SELECT COUNT(*) as cnt FROM workspace.default.silver_epa_air_quality").collect()[0]["cnt"]
print(f"Silver EPA Air Quality: {cnt} rows (after dedup & cleaning)")
display(spark.table("workspace.default.silver_epa_air_quality").limit(10))

In [0]:
# Silver Layer — NOAA Weather Observations Cleaning
# Issues found in bronze:
#   - Duplicate rows: same observation re-ingested every 20 min
#   - Some stations return all-null weather values (no current observation)
#   - Temperature in Celsius, wind speed in km/h — standardize to F and mph
#   - Timestamps stored as ISO strings with timezone offset

from pyspark.sql.functions import col, to_timestamp, trim, when, row_number, round as _round
from pyspark.sql.window import Window

bronze_noaa_obs = spark.table("bronze_noaa_weather_observations")

silver_noaa_obs = (bronze_noaa_obs
    # Filter out placeholder rows where station_id is null
    .filter(col("station_id").isNotNull())
    # Cast timestamps
    .withColumn("observation_time", to_timestamp(col("observation_time")))
    .withColumn("ingested_at", to_timestamp(col("ingested_at"), "yyyyMMdd_HHmmss"))
    # Unit conversions: C -> F, km/h -> mph
    .withColumn("temperature_f", _round(col("temperature") * 9/5 + 32, 2))
    .withColumn("temperature_c", col("temperature"))
    .withColumn("wind_speed_mph", _round(col("wind_speed") * 0.621371, 2))
    .withColumn("wind_speed_kmh", col("wind_speed"))
    # Clip humidity to valid 0-100 range
    .withColumn("humidity", when(col("humidity") > 100, 100.0).otherwise(col("humidity")))
    # Deduplicate: keep latest ingestion per station_id + observation_time
    .withColumn("rn", row_number().over(
        Window.partitionBy("station_id", "observation_time")
        .orderBy(col("ingested_at").desc())
    ))
    .filter(col("rn") == 1)
    .drop("rn")
    # Drop original unit-specific columns replaced by standardized ones
    .drop("temperature", "temperature_unit", "wind_speed", "wind_speed_unit")
    # Trim text fields
    .withColumn("station_name", trim(col("station_name")))
    .withColumn("description", trim(col("description")))
)

silver_noaa_obs.write.mode("overwrite").saveAsTable("workspace.default.silver_noaa_weather_observations")
cnt = spark.sql("SELECT COUNT(*) as cnt FROM workspace.default.silver_noaa_weather_observations").collect()[0]["cnt"]
print(f"Silver NOAA Weather Observations: {cnt} rows (after dedup & unit conversion)")
display(spark.table("workspace.default.silver_noaa_weather_observations").limit(10))

In [0]:
# Silver Layer — NOAA Weather Alerts Cleaning
# Issues found in bronze:
#   - Heavy duplication: same alert_id ingested on every 20-min cycle
#   - Timestamps (effective, expires) stored as ISO strings with timezone offset
#   - Text fields may have leading/trailing whitespace

from pyspark.sql.functions import col, to_timestamp, trim, row_number
from pyspark.sql.window import Window

bronze_noaa_alerts = spark.table("bronze_noaa_weather_alerts")

silver_noaa_alerts = (bronze_noaa_alerts
    # Filter out placeholder rows
    .filter(col("alert_id").isNotNull())
    # Cast timestamps
    .withColumn("effective", to_timestamp(col("effective")))
    .withColumn("expires", to_timestamp(col("expires")))
    .withColumn("ingested_at", to_timestamp(col("ingested_at"), "yyyyMMdd_HHmmss"))
    # Deduplicate by alert_id — keep latest ingestion
    .withColumn("rn", row_number().over(
        Window.partitionBy("alert_id")
        .orderBy(col("ingested_at").desc())
    ))
    .filter(col("rn") == 1)
    .drop("rn")
    # Standardize and trim text fields
    .withColumn("event", trim(col("event")))
    .withColumn("headline", trim(col("headline")))
    .withColumn("severity", trim(col("severity")))
    .withColumn("certainty", trim(col("certainty")))
    .withColumn("urgency", trim(col("urgency")))
    .withColumn("area_desc", trim(col("area_desc")))
)

silver_noaa_alerts.write.mode("overwrite").saveAsTable("workspace.default.silver_noaa_weather_alerts")
cnt = spark.sql("SELECT COUNT(*) as cnt FROM workspace.default.silver_noaa_weather_alerts").collect()[0]["cnt"]
print(f"Silver NOAA Weather Alerts: {cnt} rows (after dedup)")
display(spark.table("workspace.default.silver_noaa_weather_alerts").limit(10))

In [0]:
# Silver Layer — USGS Water Data Cleaning
# Issues found in bronze:
#   - Duplicate rows: same site/variable/observation re-ingested every 20 min
#   - HTML entities in variable_name (e.g., ft&#179;/s should be ft³/s)
#   - Negative streamflow values (physically impossible — sensor errors)
#   - Some observation_time values are years old (stale sensor data)
#   - Timestamps stored as ISO strings with timezone offset

from pyspark.sql.functions import col, to_timestamp, trim, regexp_replace, when, row_number
from pyspark.sql.window import Window

bronze_usgs = spark.table("bronze_usgs_water_data")

silver_usgs = (bronze_usgs
    # Filter out placeholder rows
    .filter(col("site_code").isNotNull())
    # Clean HTML entities in variable_name
    .withColumn("variable_name", regexp_replace(col("variable_name"), "&#179;", "³"))
    .withColumn("variable_name", regexp_replace(col("variable_name"), "&amp;", "&"))
    # Cast timestamps
    .withColumn("observation_time", to_timestamp(col("observation_time")))
    .withColumn("ingested_at", to_timestamp(col("ingested_at"), "yyyyMMdd_HHmmss"))
    # Null out negative streamflow values (sensor error — discharge cannot be negative)
    .withColumn("value",
        when((col("variable_code") == "00060") & (col("value") < 0), None)
        .otherwise(col("value"))
    )
    # Deduplicate: keep latest ingestion per site_code + variable_code + observation_time
    .withColumn("rn", row_number().over(
        Window.partitionBy("site_code", "variable_code", "observation_time")
        .orderBy(col("ingested_at").desc())
    ))
    .filter(col("rn") == 1)
    .drop("rn")
    # Trim text fields
    .withColumn("site_name", trim(col("site_name")))
    .withColumn("unit", trim(col("unit")))
)

silver_usgs.write.mode("overwrite").saveAsTable("workspace.default.silver_usgs_water_data")
cnt = spark.sql("SELECT COUNT(*) as cnt FROM workspace.default.silver_usgs_water_data").collect()[0]["cnt"]
print(f"Silver USGS Water Data: {cnt} rows (after dedup, HTML entity cleaning & null invalid values)")
display(spark.table("workspace.default.silver_usgs_water_data").limit(10))

In [0]:
# Silver Layer — NCDOT Crash Data Cleaning
# Issues found in bronze:
#   - Duplicate rows: same crash re-ingested every 20 min
#   - Timestamps stored as strings (start_datetime, end_datetime, last_update)
#   - is_full_closure stored as string ("Y"/"N") instead of boolean
#   - Text fields may have whitespace; road names need standardization
#   - Some records missing lat/lon coordinates

from pyspark.sql.functions import col, to_timestamp, trim, when, row_number
from pyspark.sql.window import Window

bronze_crash = spark.table("bronze_ncdot_crash_data")

silver_crash = (bronze_crash
    # Filter out placeholder rows
    .filter(col("object_id").isNotNull())
    # Cast timestamps from strings
    .withColumn("start_datetime", to_timestamp(col("start_datetime")))
    .withColumn("end_datetime", to_timestamp(col("end_datetime")))
    .withColumn("last_update", to_timestamp(col("last_update")))
    .withColumn("ingested_at", to_timestamp(col("ingested_at"), "yyyyMMdd_HHmmss"))
    # Convert is_full_closure from string to boolean
    .withColumn("is_full_closure",
        when(trim(col("is_full_closure")) == "Y", True)
        .when(trim(col("is_full_closure")) == "N", False)
        .otherwise(None)
    )
    # Deduplicate by object_id — keep latest ingestion
    .withColumn("rn", row_number().over(
        Window.partitionBy("object_id")
        .orderBy(col("ingested_at").desc())
    ))
    .filter(col("rn") == 1)
    .drop("rn")
    # Trim and standardize text fields
    .withColumn("road", trim(col("road")))
    .withColumn("reason", trim(col("reason")))
    .withColumn("condition", trim(col("condition")))
    .withColumn("direction", trim(col("direction")))
    .withColumn("route_type", trim(col("route_type")))
    .withColumn("county_name", trim(col("county_name")))
    .withColumn("location", trim(col("location")))
    # Filter out records without valid coordinates
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())
)

silver_crash.write.mode("overwrite").saveAsTable("workspace.default.silver_ncdot_crash_data")
cnt = spark.sql("SELECT COUNT(*) as cnt FROM workspace.default.silver_ncdot_crash_data").collect()[0]["cnt"]
print(f"Silver NCDOT Crash Data: {cnt} rows (after dedup, boolean conversion & coordinate filtering)")
display(spark.table("workspace.default.silver_ncdot_crash_data").limit(10))

In [0]:
# Silver Layer — NCGS Landslide Data Cleaning
# Issues found in bronze:
#   - Duplicate rows: same landslide re-ingested on every pipeline run
#   - Timestamps stored as strings (ingested_at)
#   - Some records missing lat/lon coordinates
#   - Text fields may have whitespace

from pyspark.sql.functions import col, to_timestamp, trim, row_number
from pyspark.sql.window import Window

bronze_landslide = spark.table("bronze_ncgs_landslide_data")

silver_landslide = (bronze_landslide
    # Filter out placeholder rows
    .filter(col("object_id").isNotNull())
    # Cast timestamps
    .withColumn("ingested_at", to_timestamp(col("ingested_at"), "yyyyMMdd_HHmmss"))
    # Deduplicate by object_id — keep latest ingestion
    .withColumn("rn", row_number().over(
        Window.partitionBy("object_id")
        .orderBy(col("ingested_at").desc())
    ))
    .filter(col("rn") == 1)
    .drop("rn")
    # Trim text fields
    .withColumn("impact", trim(col("impact")))
    .withColumn("damage", trim(col("damage")))
    .withColumn("associated_storm", trim(col("associated_storm")))
    .withColumn("comments", trim(col("comments")))
    .withColumn("site_visit", trim(col("site_visit")))
    # Filter out records without valid coordinates
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())
)

silver_landslide.write.mode("overwrite").saveAsTable("workspace.default.silver_ncgs_landslide_data")
cnt = spark.sql("SELECT COUNT(*) as cnt FROM workspace.default.silver_ncgs_landslide_data").collect()[0]["cnt"]
print(f"Silver NCGS Landslide Data: {cnt} rows (after dedup & coordinate filtering)")
display(spark.table("workspace.default.silver_ncgs_landslide_data").limit(10))

In [0]:
# Silver Layer — NCEM Flood Zone Data Cleaning
# Issues found in bronze:
#   - Duplicate rows: same flood zone re-ingested on every pipeline run
#   - Timestamps stored as strings (ingested_at)
#   - sfha stored as integer (0/1) — standardize to boolean
#   - zone_type needs trimming and standardization
#   - Some records may have null coordinates (failed centroid computation)

from pyspark.sql.functions import col, to_timestamp, trim, when, row_number
from pyspark.sql.window import Window

bronze_flood = spark.table("bronze_ncem_flood_zones")

silver_flood = (bronze_flood
    # Filter out placeholder rows
    .filter(col("object_id").isNotNull())
    # Cast timestamps
    .withColumn("ingested_at", to_timestamp(col("ingested_at"), "yyyyMMdd_HHmmss"))
    # Convert sfha from integer to boolean
    .withColumn("sfha", when(col("sfha") == 1, True).otherwise(False))
    # Deduplicate by object_id — keep latest ingestion
    .withColumn("rn", row_number().over(
        Window.partitionBy("object_id")
        .orderBy(col("ingested_at").desc())
    ))
    .filter(col("rn") == 1)
    .drop("rn")
    # Trim and standardize text fields
    .withColumn("zone_type", trim(col("zone_type")))
    .withColumn("zone_subtype", trim(col("zone_subtype")))
    # Filter out records without valid coordinates
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())
)

silver_flood.write.mode("overwrite").saveAsTable("workspace.default.silver_ncem_flood_zones")
cnt = spark.sql("SELECT COUNT(*) as cnt FROM workspace.default.silver_ncem_flood_zones").collect()[0]["cnt"]
print(f"Silver NCEM Flood Zones: {cnt} rows (after dedup, boolean conversion & coordinate filtering)")
display(spark.table("workspace.default.silver_ncem_flood_zones").limit(10))

In [0]:
# Gold Layer — Curated Analytics Tables
# Aggregates and latest-state views from cleaned silver tables.
# Designed for dashboards, reporting, and downstream ML features.

from pyspark.sql.functions import (
    col, desc, row_number, count as _count, max as _max,
    when, current_timestamp
)
from pyspark.sql.window import Window

# 1. Gold — Latest air quality reading per location
w_aq = Window.partitionBy("latitude", "longitude").orderBy(desc("measurement_time"))
gold_air_quality = (spark.table("workspace.default.silver_epa_air_quality")
    .withColumn("rn", row_number().over(w_aq))
    .filter(col("rn") == 1)
    .drop("rn")
    .select("latitude", "longitude", "us_aqi", "aqi_category",
            "pm2_5", "pm10", "o3", "no2", "so2", "co", "measurement_time")
)
gold_air_quality.write.mode("overwrite").saveAsTable("workspace.default.gold_air_quality_latest")
print(f"gold_air_quality_latest: {gold_air_quality.count()} rows")

# 2. Gold — Latest weather conditions per station
w_wx = Window.partitionBy("station_id").orderBy(desc("observation_time"))
gold_weather = (spark.table("workspace.default.silver_noaa_weather_observations")
    .withColumn("rn", row_number().over(w_wx))
    .filter(col("rn") == 1)
    .drop("rn")
    .select("station_id", "station_name", "temperature_f", "temperature_c",
            "wind_speed_mph", "wind_speed_kmh", "humidity",
            "barometric_pressure", "wind_direction", "observation_time")
)
gold_weather.write.mode("overwrite").saveAsTable("workspace.default.gold_weather_summary")
print(f"gold_weather_summary: {gold_weather.count()} rows")

# 3. Gold — Active weather alerts (expires > now)
gold_alerts = (spark.table("workspace.default.silver_noaa_weather_alerts")
    .filter(col("expires") > current_timestamp())
    .select("alert_id", "event", "headline", "severity", "certainty",
            "urgency", "area_desc", "effective", "expires")
)
gold_alerts.write.mode("overwrite").saveAsTable("workspace.default.gold_active_weather_alerts")
print(f"gold_active_weather_alerts: {gold_alerts.count()} rows")

# 4. Gold — Latest water reading per site + variable
w_h2o = Window.partitionBy("site_code", "variable_code").orderBy(desc("observation_time"))
gold_water = (spark.table("workspace.default.silver_usgs_water_data")
    .withColumn("rn", row_number().over(w_h2o))
    .filter(col("rn") == 1)
    .drop("rn")
    .select("site_code", "site_name", "variable_code", "variable_name",
            "value", "unit", "latitude", "longitude", "observation_time")
)
gold_water.write.mode("overwrite").saveAsTable("workspace.default.gold_water_conditions")
print(f"gold_water_conditions: {gold_water.count()} rows")

# 5. Gold — Crash summary by county and route type
gold_crash = (spark.table("workspace.default.silver_ncdot_crash_data")
    .groupBy("county_name", "route_type")
    .agg(
        _count("*").alias("crash_count"),
        _count(when(col("is_full_closure") == True, 1)).alias("full_closure_count"),
        _max("start_datetime").alias("most_recent_crash")
    )
    .orderBy(desc("crash_count"))
)
gold_crash.write.mode("overwrite").saveAsTable("workspace.default.gold_crash_summary")
print(f"gold_crash_summary: {gold_crash.count()} rows")

# 6. Gold — Landslide summary by storm and impact type
gold_landslide = (spark.table("workspace.default.silver_ncgs_landslide_data")
    .groupBy("associated_storm", "impact")
    .agg(
        _count("*").alias("landslide_count"),
        _max("fatalities").alias("max_fatalities"),
        _max("latitude").alias("sample_lat"),
        _max("longitude").alias("sample_lon")
    )
    .orderBy(desc("landslide_count"))
)
gold_landslide.write.mode("overwrite").saveAsTable("workspace.default.gold_landslide_summary")
print(f"gold_landslide_summary: {gold_landslide.count()} rows")

# 7. Gold — Flood zone summary by zone type and county
gold_flood = (spark.table("workspace.default.silver_ncem_flood_zones")
    .groupBy("zone_type", "county_fips")
    .agg(
        _count("*").alias("zone_count"),
        _max("depth").alias("max_depth"),
        _max("static_bfe").alias("max_bfe")
    )
    .orderBy(desc("zone_count"))
)
gold_flood.write.mode("overwrite").saveAsTable("workspace.default.gold_flood_zone_summary")
print(f"gold_flood_zone_summary: {gold_flood.count()} rows")
display(gold_flood.limit(20))

## Medallion Pipeline Summary

| Layer | Table | Description |
| --- | --- | --- |
| **Bronze** | `epa_air_quality`, `noaa_weather_observations`, `noaa_weather_alerts`, `usgs_water_data`, `ncdot_crash_data`, `ncgs_landslide_data`, `ncem_flood_zones` | Raw data from ingestion (unchanged) |
| **Silver** | `silver_epa_air_quality` | Deduped, timestamps cast, AQI category derived |
| **Silver** | `silver_noaa_weather_observations` | Deduped, C→F & km/h→mph conversions, humidity clipped |
| **Silver** | `silver_noaa_weather_alerts` | Deduped by alert_id, timestamps cast, text trimmed |
| **Silver** | `silver_usgs_water_data` | Deduped, HTML entities cleaned, negative streamflow nulled |
| **Silver** | `silver_ncdot_crash_data` | Deduped, is_full_closure→boolean, timestamps cast, coords filtered |
| **Silver** | `silver_ncgs_landslide_data` | Deduped by object_id, timestamps cast, text trimmed, coords filtered |
| **Silver** | `silver_ncem_flood_zones` | Deduped by object_id, sfha→boolean, zone_type trimmed, coords filtered |
| **Gold** | `gold_air_quality_latest` | Latest AQI per location |
| **Gold** | `gold_weather_summary` | Latest weather per station |
| **Gold** | `gold_active_weather_alerts` | Currently active alerts only |
| **Gold** | `gold_water_conditions` | Latest water reading per site/variable |
| **Gold** | `gold_crash_summary` | Crashes aggregated by county & route type |
| **Gold** | `gold_landslide_summary` | Landslides aggregated by storm & impact type |
| **Gold** | `gold_flood_zone_summary` | Flood zones aggregated by zone type & county |